In [0]:
MY_ID = "2328073"

CATALOG = "workspace"
SCHEMA = "dbsf_2328073"
VOL = "/Volumes/workspace/dbsf_2328073/raw_files"

print("MY_ID:", MY_ID)
print("SCHEMA:", SCHEMA)
print("VOL:", VOL)

MY_ID: 2328073
SCHEMA: dbsf_2328073
VOL: /Volumes/workspace/dbsf_2328073/raw_files


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================================================
# Helper functions
# =========================================================

def bronze(table_name):
    return spark.table(
        f"workspace.dbsf_2328073.bronze_{table_name}_2328073"
    )


def silver_table(table_name):
    return f"workspace.dbsf_2328073.silver_{table_name}_2328073"


# =========================================================
# 1. ORDERS
# =========================================================

orders_bronze = bronze("orders")

print("Raw orders:", orders_bronze.count())


# Convert timestamps safely using try_cast
orders_typed = orders_bronze.selectExpr(
    "*",

    "try_cast(order_purchase_timestamp AS TIMESTAMP) "
    "AS purchase_ts",

    "try_cast(order_approved_at AS TIMESTAMP) "
    "AS approved_ts",

    "try_cast(order_delivered_carrier_date AS TIMESTAMP) "
    "AS carrier_ts",

    "try_cast(order_delivered_customer_date AS TIMESTAMP) "
    "AS delivered_ts",

    "try_cast(order_estimated_delivery_date AS TIMESTAMP) "
    "AS estimated_delivery_ts"
)


# Deduplicate by order_id
# Latest ingestion wins; row_hash breaks ties.
order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("_ingested_at").desc(),
        F.col("_row_hash").asc()
    )
)

orders_silver = (
    orders_typed
    .withColumn("_rn", F.row_number().over(order_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


# Delivery duration
orders_silver = orders_silver.withColumn(
    "delivery_days",
    F.datediff(
        F.col("delivered_ts"),
        F.col("purchase_ts")
    )
)


# Late-delivery definition
# NULL delivery date = undelivered, NOT late.
orders_silver = orders_silver.withColumn(
    "late_flag",
    F.when(
        F.col("delivered_ts").isNotNull()
        & F.col("estimated_delivery_ts").isNotNull()
        & (
            F.col("delivered_ts")
            > F.col("estimated_delivery_ts")
        ),
        F.lit(1)
    ).otherwise(F.lit(0))
)


orders_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        silver_table("orders")
    )


print(
    "Silver orders:",
    orders_silver.count()
)


# =========================================================
# 2. SELLERS
# =========================================================

sellers_bronze = bronze("sellers")

sellers_silver = (
    sellers_bronze
    .select(
        "seller_id",
        "seller_city",
        "seller_state"
    )
    .dropDuplicates(["seller_id"])
)

sellers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        silver_table("sellers")
    )

print(
    "Silver sellers:",
    sellers_silver.count()
)


# =========================================================
# 3. ORDER ITEMS
# =========================================================

items_bronze = bronze("order_items")

items_typed = items_bronze.selectExpr(
    "*",

    "try_cast(order_item_id AS INT) "
    "AS order_item_num",

    "try_cast(shipping_limit_date AS TIMESTAMP) "
    "AS shipping_limit_ts",

    "try_cast(price AS DOUBLE) "
    "AS unit_price",

    "try_cast(freight_value AS DOUBLE) "
    "AS freight_value_num"
)


# Find items whose seller_id does not exist
# in the seller master.
orphan_items = (
    items_typed
    .join(
        sellers_silver.select("seller_id"),
        on="seller_id",
        how="left_anti"
    )
    .withColumn(
        "reject_reason",
        F.lit("ORPHAN_SELLER_ID")
    )
)


orphan_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        silver_table("rejects")
    )


# Keep only valid seller records
valid_items = (
    items_typed
    .join(
        sellers_silver.select("seller_id"),
        on="seller_id",
        how="inner"
    )
)


valid_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        silver_table("order_items")
    )


print(
    "Silver order items:",
    valid_items.count()
)

print(
    "Rejected orphan items:",
    orphan_items.count()
)


# =========================================================
# 4. REVIEWS
# =========================================================

reviews_bronze = bronze("order_reviews")

reviews_typed = reviews_bronze.selectExpr(
    "review_id",
    "order_id",
    "try_cast(review_score AS INT) AS review_score"
)


# One business record per order.
# Average the review score if an order has
# more than one review record.
review_order = (
    reviews_typed
    .groupBy("order_id")
    .agg(
        F.avg("review_score")
        .alias("avg_review_score"),

        F.count("*")
        .alias("review_records")
    )
)


review_order.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        silver_table("order_reviews")
    )


print(
    "Silver order reviews:",
    review_order.count()
)


# =========================================================
# 5. CUSTOMERS
# =========================================================

customers_bronze = bronze("customers")

customers_silver = (
    customers_bronze
    .select(
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates(["customer_id"])
)


customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        silver_table("customers")
    )


print(
    "Silver customers:",
    customers_silver.count()
)


# =========================================================
# 6. PRODUCTS + CATEGORY TRANSLATION
# =========================================================

products_bronze = bronze("products")

translation_bronze = bronze(
    "category_translation"
)


products_silver = (
    products_bronze
    .join(
        translation_bronze,
        on="product_category_name",
        how="left"
    )
    .select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
)


products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        silver_table("products")
    )


print(
    "Silver products:",
    products_silver.count()
)


print("\n================================")
print("SILVER BASIC CLEANING COMPLETE")
print("================================")

Raw orders: 99441
Silver orders: 99441
Silver sellers: 3095
Silver order items: 112650
Rejected orphan items: 0
Silver order reviews: 99743
Silver customers: 99441
Silver products: 32951

SILVER BASIC CLEANING COMPLETE


In [0]:
from pyspark.sql import functions as F

print("======================================")
print("SILVER DATA QUALITY CHECKS")
print("======================================")

# --------------------------------------------------
# 1. Duplicate orders
# --------------------------------------------------

duplicate_orders = (
    spark.table(
        "workspace.dbsf_2328073.silver_orders_2328073"
    )
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate order IDs:", duplicate_orders)


# --------------------------------------------------
# 2. Invalid price conversions
# --------------------------------------------------

items_check = spark.table(
    "workspace.dbsf_2328073.silver_order_items_2328073"
)

invalid_prices = items_check.filter(
    F.col("unit_price").isNull()
    & F.col("price").isNotNull()
).count()

print("Invalid price conversions:", invalid_prices)


# --------------------------------------------------
# 3. Invalid freight conversions
# --------------------------------------------------

invalid_freight = items_check.filter(
    F.col("freight_value_num").isNull()
    & F.col("freight_value").isNotNull()
).count()

print("Invalid freight conversions:", invalid_freight)


# --------------------------------------------------
# 4. Missing purchase timestamps
# --------------------------------------------------

orders_check = spark.table(
    "workspace.dbsf_2328073.silver_orders_2328073"
)

missing_purchase = orders_check.filter(
    F.col("purchase_ts").isNull()
).count()

print("Missing purchase timestamp:", missing_purchase)


# --------------------------------------------------
# 5. Missing delivered timestamps
# --------------------------------------------------

missing_delivered = orders_check.filter(
    F.col("delivered_ts").isNull()
).count()

print("Missing delivered timestamp:", missing_delivered)


# --------------------------------------------------
# 6. Late orders
# --------------------------------------------------

late_orders = orders_check.filter(
    F.col("late_flag") == 1
).count()

print("Late orders:", late_orders)


# --------------------------------------------------
# 7. Undelivered orders incorrectly marked late
# --------------------------------------------------

bad_late_flags = orders_check.filter(
    F.col("delivered_ts").isNull()
    & (F.col("late_flag") == 1)
).count()

print("Undelivered orders marked late:", bad_late_flags)


print("======================================")
print("SILVER VALIDATION COMPLETE")
print("======================================")

SILVER DATA QUALITY CHECKS
Duplicate order IDs: 0
Invalid price conversions: 0
Invalid freight conversions: 0
Missing purchase timestamp: 0
Missing delivered timestamp: 2965
Late orders: 7827
Undelivered orders marked late: 0
SILVER VALIDATION COMPLETE


In [0]:
orders_s = spark.table(
    "workspace.dbsf_2328073.silver_orders_2328073"
)

reviews_s = spark.table(
    "workspace.dbsf_2328073.silver_order_reviews_2328073"
)

review_orphans = (
    reviews_s
    .join(
        orders_s.select("order_id"),
        on="order_id",
        how="left_anti"
    )
)

print(
    "Review records whose order_id is not in Silver orders:",
    review_orphans.count()
)

Review records whose order_id is not in Silver orders: 1070


In [0]:
reviews_s = (
    reviews_s
    .join(
        orders_s.select("order_id"),
        on="order_id",
        how="inner"
    )
)

reviews_s.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.dbsf_2328073.silver_order_reviews_2328073"
    )

print(
    "Clean Silver review rows:",
    reviews_s.count()
)

Clean Silver review rows: 98673


In [0]:
from pyspark.sql import functions as F

# =========================================================
# Load Silver tables
# =========================================================

orders_s = spark.table(
    "workspace.dbsf_2328073.silver_orders_2328073"
)

items_s = spark.table(
    "workspace.dbsf_2328073.silver_order_items_2328073"
)

products_s = spark.table(
    "workspace.dbsf_2328073.silver_products_2328073"
)

customers_s = spark.table(
    "workspace.dbsf_2328073.silver_customers_2328073"
)

reviews_s = spark.table(
    "workspace.dbsf_2328073.silver_order_reviews_2328073"
)

sellers_s = spark.table(
    "workspace.dbsf_2328073.silver_sellers_2328073"
)


# =========================================================
# 1. Revenue at order level
# =========================================================

order_revenue = (
    items_s
    .groupBy("order_id")
    .agg(
        F.sum("unit_price").alias("order_revenue"),
        F.count("*").alias("item_lines")
    )
)


# =========================================================
# 2. Main order-level business table
# =========================================================

order_metrics = (
    orders_s
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "purchase_ts",
        "delivered_ts",
        "estimated_delivery_ts",
        "delivery_days",
        "late_flag"
    )

    .join(
        customers_s.select(
            "customer_id",
            "customer_unique_id"
        ),
        on="customer_id",
        how="left"
    )

    .join(
        order_revenue,
        on="order_id",
        how="left"
    )

    .join(
        reviews_s.select(
            "order_id",
            "avg_review_score"
        ),
        on="order_id",
        how="left"
    )
)


order_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.dbsf_2328073.silver_order_metrics_2328073"
    )


print(
    "Order metrics:",
    order_metrics.count()
)


# =========================================================
# 3. Product category at order level
# =========================================================

items_products = (
    items_s

    .join(
        products_s.select(
            "product_id",
            "product_category_name",
            "product_category_name_english"
        ),
        on="product_id",
        how="left"
    )

    .withColumn(
        "category",
        F.coalesce(
            F.col("product_category_name_english"),
            F.col("product_category_name"),
            F.lit("unknown")
        )
    )
)


order_category = (
    items_products
    .groupBy(
        "order_id",
        "category"
    )
    .agg(
        F.sum("unit_price")
        .alias("category_revenue"),

        F.count("*")
        .alias("category_item_lines")
    )
)


order_category.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.dbsf_2328073.silver_order_category_2328073"
    )


print(
    "Order-category rows:",
    order_category.count()
)


# =========================================================
# 4. Seller + order business grain
# =========================================================

seller_order = (
    items_s

    .groupBy(
        "seller_id",
        "order_id"
    )

    .agg(
        F.sum("unit_price")
        .alias("seller_revenue")
    )

    .join(
        order_metrics.select(
            "order_id",
            "order_status",
            "late_flag",
            "avg_review_score"
        ),
        on="order_id",
        how="left"
    )
)


seller_order.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.dbsf_2328073.silver_seller_order_2328073"
    )


print(
    "Seller-order rows:",
    seller_order.count()
)


print("\n======================================")
print("ANALYTICAL SILVER TABLES COMPLETE")
print("======================================")

Order metrics: 99441
Order-category rows: 99470
Seller-order rows: 100010

ANALYTICAL SILVER TABLES COMPLETE
